In [ ]:
# E2 notebook (run in parallel with colab_run.ipynb; CPU runtime)
# 1) Drive + secrets (Colab sidebar 🔑: add KAGGLE_API_TOKEN and GH_TOKEN, enable notebook access)
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['DRV'] = DRV = '/content/drive/MyDrive/s6e10'
os.environ['S6_DATA'] = '/content/data'           # fast local disk
os.environ['S6_ART'] = f'{DRV}/artifacts'         # oof/, test/, ckpt/, experiments.csv live on Drive -> resumable
os.makedirs(os.environ['S6_ART'], exist_ok=True)

In [ ]:
# 2) Code
!test -d /content/s6e10 || git clone -q https://$GH_TOKEN@github.com/0-Playor-0/s6e10-airline.git /content/s6e10
!cd /content/s6e10 && git pull -q && git log -1 --oneline
!pip install -q -U kaggle pyyaml lightgbm

In [ ]:
%%bash
# 3) Data for E2 only (comp + original). Reads the Drive cache if the main notebook already filled it,
#    otherwise downloads straight to local disk; never writes the shared cache, so both notebooks can run at once.
set -e
C=$DRV/cache; mkdir -p /content/data/comp /content/data/orig
if [ -f $C/comp/train.csv ]; then rsync -a $C/comp/ /content/data/comp/; else
  kaggle competitions download -c playground-series-s6e10 -p /content/data/comp -q && cd /content/data/comp && unzip -oq '*.zip' && rm -f *.zip; fi
if [ -f $C/orig/train.csv ]; then rsync -a $C/orig/ /content/data/orig/; else
  kaggle datasets download teejmahal20/airline-passenger-satisfaction -p /content/data/orig -q --unzip; fi
ls /content/data/comp /content/data/orig

In [ ]:
# 4) E2 own LightGBM: route profile + original-data teacher + fold-safe TE, our folds; ~1 h, resumable per fold
!cd /content/s6e10 && python src/run.py --exp e2_lgb